In [ ]:
!pip install requests pandas numpy

In [ ]:
import requests
import time
import pandas as pd
import numpy as np
import json
import hmac
import hashlib
import urllib.parse
from datetime import datetime
import base64

class KorbitTrader:
    def __init__(self, api_key, api_secret):
        self.api_key = api_key
        self.api_secret = api_secret
        self.base_url = "https://api.korbit.co.kr"

        # 거래 설정
        self.currency_pair = "btc_krw"
        self.min_order_amount = 5000  # 최소 주문 금액 (KRW)
        self.max_investment = 50000   # 최대 투자 금액 (KRW)
        self.profit_target = 0.02     # 2% 수익률 목표
        self.stop_loss = 0.01         # 1% 손절매

        # 거래 기록 (구현 예시, 실제 활용시 DB 연동 등 고려)
        self.positions = []
        self.trade_history = []

    def get_timestamp(self):
        """현재 시간을 밀리초 단위 타임스탬프로 반환"""
        return str(int(time.time() * 1000))

    def create_signature(self, params_string):
        """HMAC-SHA256 서명 생성"""
        signature = hmac.new(
            self.api_secret.encode('utf-8'),
            params_string.encode('utf-8'),
            hashlib.sha256
        ).hexdigest()
        return signature

    def get_headers(self, params_string=None):
        """인증 헤더 생성 (주로 POST 요청 시 사용될 수 있음)"""
        headers = {
            'Accept': 'application/json',
            'Content-Type': 'application/x-www-form-urlencoded'
        }
        if params_string: # 이 로직은 현재 직접 사용되지 않고, 각 요청 메서드에서 헤더를 개별 구성.
            headers['X-KAPI-KEY'] = self.api_key
        return headers

    def test_connection(self):
        """API 연결 테스트"""
        try:
            # 공개 API 먼저 테스트
            public_url = f"{self.base_url}/v2/tickers"
            params = {'symbol': self.currency_pair}
            response = requests.get(public_url, params=params)

            if response.status_code == 200:
                print("✅ 공개 API 연결 성공")

                # 인증 API 테스트 (잔고 조회)
                timestamp = self.get_timestamp()
                params_data_string = f'timestamp={timestamp}'
                signature = self.create_signature(params_data_string)

                private_url = f"{self.base_url}/v2/balance?{params_data_string}&signature={signature}"
                headers = {
                    'X-KAPI-KEY': self.api_key,
                    'Accept': 'application/json' # GET 요청 시 Content-Type은 보통 불필요
                }

                auth_response = requests.get(private_url, headers=headers)

                if auth_response.status_code == 200:
                    print("✅ 인증 API 연결 성공")
                    return True
                else:
                    print(f"❌ 인증 API 연결 실패: {auth_response.status_code}")
                    print(f"응답: {auth_response.text}")
                    return False
            else:
                print(f"❌ 공개 API 연결 실패: {response.status_code}")
                print(f"응답: {response.text}")
                return False

        except Exception as e:
            print(f"❌ 연결 테스트 중 오류 발생: {e}")
            return False

    def get_balance(self):
        """잔고 조회"""
        timestamp = self.get_timestamp()
        params_data_string = f'timestamp={timestamp}' # 서명 대상 문자열, API 명세 확인 필요
        signature = self.create_signature(params_data_string)

        url = f"{self.base_url}/v2/balance?{params_data_string}&signature={signature}"
        headers = {
            'X-KAPI-KEY': self.api_key,
            'Accept': 'application/json'
        }

        try:
            response = requests.get(url, headers=headers)
            if response.status_code == 200:
                return response.json().get('data', []) # data 키가 없을 경우 빈 리스트 반환
            else:
                print(f"❌ 잔고 조회 실패: {response.status_code}")
                print(f"응답: {response.text}")
                return None
        except Exception as e:
            print(f"❌ 잔고 조회 중 오류 발생: {e}")
            return None

    def get_ticker(self):
        """현재가 조회 (공개 API) - 수정됨"""
        url = f"{self.base_url}/v2/tickers"
        params = {'symbol': self.currency_pair}

        try:
            response = requests.get(url, params=params)
            if response.status_code == 200:
                data = response.json()

                # API 응답 구조 확인: data['data']가 리스트이고, 그 안에 티커 객체가 있는 경우
                if data.get('success') and isinstance(data.get('data'), list) and len(data['data']) > 0:
                    actual_ticker_data = data['data'][0] # 리스트의 첫 번째 요소를 사용
                    # 추가적으로 symbol 확인 (선택 사항, 이미 params로 요청했으므로)
                    if actual_ticker_data.get('symbol') == self.currency_pair:
                        # print("DEBUG: 실제 티커 데이터 (data['data'][0]):", actual_ticker_data) # 필요시 이 print문 활성화
                        return actual_ticker_data
                    else:
                        print(f"❌ 요청한 symbol({self.currency_pair})과 응답의 symbol({actual_ticker_data.get('symbol')})이 다릅니다.")
                        return None
                else:
                    print("❌ 현재가 데이터가 없거나 예상된 형식이 아닙니다. (success 플래그 또는 data 필드 구조 문제)")
                    print(f"DEBUG: API 응답 data: {data}") # 상세 응답 내용 다시 확인
                    return None
            else:
                print(f"❌ 현재가 조회 실패: {response.status_code}")
                print(f"응답: {response.text}")
                return None
        except Exception as e:
            print(f"❌ 현재가 조회 중 오류 발생: {e}")
            import traceback
            traceback.print_exc()
            return None

    def get_orderbook(self):
        """호가 조회 (공개 API)"""
        url = f"{self.base_url}/v2/orderbook"
        params = {'symbol': self.currency_pair}
        try:
            response = requests.get(url, params=params)
            if response.status_code == 200:
                return response.json() # API 응답 구조 확인 필요
            else:
                print(f"❌ 호가 조회 실패: {response.status_code}")
                print(f"응답: {response.text}")
                return None
        except Exception as e:
            print(f"❌ 호가 조회 중 오류 발생: {e}")
            return None

    def place_order(self, side, qty, price=None):
        """주문 실행 - Korbit API v2 POST 요청 서명 방식 및 파라미터 확인 필수"""
        timestamp = self.get_timestamp()

        params_for_signature = {
            'symbol': self.currency_pair,
            'side': side,
            'qty': str(qty),
            'timestamp': timestamp
            # Korbit API v2 문서에서 nonce 또는 client_id 등의 추가 파라미터가 필요한지 확인
        }

        if price:
            params_for_signature['price'] = str(price)
            params_for_signature['orderType'] = 'limit' # API 문서에서 'type' 등 다른 이름일 수 있음
        else:
            params_for_signature['orderType'] = 'market'
        params_string = urllib.parse.urlencode(params_for_signature)
        signature = self.create_signature(params_string)

        request_params = params_for_signature.copy()
        request_params['signature'] = signature

        url = f"{self.base_url}/v2/orders"
        headers = {
            'X-KAPI-KEY': self.api_key,
            'Content-Type': 'application/x-www-form-urlencoded',
            'Accept': 'application/json'
        }

        try:
            response = requests.post(url, data=request_params, headers=headers)
            if response.status_code == 200 or response.status_code == 201:
                order_result = response.json()
                print(f"✅ 주문 성공: {side} {qty} BTC ({params_for_signature['orderType']})")
                self.trade_history.append({'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'), **order_result})
                return order_result
            else:
                print(f"❌ 주문 실패: {response.status_code}")
                print(f"요청 파라미터: {request_params}")
                print(f"응답: {response.text}")
                return None
        except Exception as e:
            print(f"❌ 주문 중 오류 발생: {e}")
            return None

    def get_market_data(self, limit=100):
        """시장 데이터 분석을 위한 최근 체결 내역 조회 (공개 API)"""
        url = f"{self.base_url}/v2/trades"
        params = {
            'symbol': self.currency_pair,
            'limit': limit # API가 'limit' 파라미터를 지원하는지, 이름이 다른지 확인
        }
        try:
            response = requests.get(url, params=params)
            if response.status_code == 200:
                data = response.json()
                if isinstance(data, list): # 응답이 체결내역 리스트 자체인 경우
                    return data
                elif data.get('success') and isinstance(data.get('data'), list):
                     return data['data']
                else:
                    print("❌ 최근 체결 데이터가 없거나 예상된 형식이 아닙니다.")
                    print(f"응답: {data}")
                    return None
            else:
                print(f"❌ 최근 체결 데이터 조회 실패: {response.status_code}")
                print(f"응답: {response.text}")
                return None
        except Exception as e:
            print(f"❌ 최근 체결 데이터 조회 중 오류 발생: {e}")
            return None

    def simple_strategy(self): # 수정됨
        """간단한 거래 전략 (이동평균 기반)"""
        market_data = self.get_market_data(50) # 최근 50건 거래내역
        if not market_data or len(market_data) < 10: # 최소 10개 데이터 필요 가정
            print("⚠️ 시장 데이터 부족으로 전략 실행 불가")
            return None

        try:
            prices = [float(trade['price']) for trade in market_data]
            df = pd.DataFrame({'price': prices})
        except (KeyError, TypeError) as e: # 'price' 키가 없거나, trade가 dict가 아닌 경우
            print(f"❌ 시장 데이터(거래 내역) 처리 중 오류: {e}")
            return None
        except ValueError as e:
            print(f"❌ 시장 데이터(거래 내역)의 가격을 숫자로 변환할 수 없습니다: {e}")
            return None

        df['sma_5'] = df['price'].rolling(window=5).mean()
        df['sma_10'] = df['price'].rolling(window=10).mean()

        ticker_data = self.get_ticker()
        if not ticker_data:
            return None

        try:
            current_price_str = ticker_data.get('close') # 'close' 키가 없을 수 있으므로 .get() 사용
            if current_price_str is None:
                print(f"❌ Ticker 데이터에서 'close' 키를 찾을 수 없습니다. 받은 데이터: {ticker_data}")
                return None
            current_price = float(current_price_str)
        except ValueError:
            print(f"❌ Ticker 데이터의 'close' 값을 숫자로 변환할 수 없습니다. 값: {current_price_str}")
            return None
        except Exception as e: # 기타 예외 처리
            print(f"❌ Ticker 데이터 처리 중 알 수 없는 오류: {e}")
            return None


        # 데이터가 충분하지 않으면 이동평균이 NaN일 수 있음. 이 경우 현재가로 대체.
        sma_5 = df['sma_5'].iloc[-1] if not pd.isna(df['sma_5'].iloc[-1]) else current_price
        sma_10 = df['sma_10'].iloc[-1] if not pd.isna(df['sma_10'].iloc[-1]) else current_price

        print(f"현재가: {current_price:,.0f}원")
        print(f"5일 평균: {sma_5:,.0f}원 (데이터 수: {len(df)})")
        print(f"10일 평균: {sma_10:,.0f}원 (데이터 수: {len(df)})")

        if sma_5 > sma_10 and current_price > sma_5:
            return "BUY"
        elif sma_5 < sma_10 and current_price < sma_5:
            return "SELL"
        else:
            return "HOLD"

    def calculate_position_size(self, price_per_btc):
        """포지션 크기(BTC 수량) 계산"""
        balance_data = self.get_balance()
        if not balance_data:
            print("⚠️ 잔고 조회 실패로 포지션 크기 계산 불가")
            return 0

        krw_balance = 0
        for account in balance_data:
            if account.get('currency') == 'krw':
                try:
                    krw_balance = float(account.get('available', 0))
                except ValueError:
                    print(f"⚠️ KRW 잔고 값 변환 오류: {account.get('available')}")
                    krw_balance = 0
                break

        print(f"현재 KRW 잔고: {krw_balance:,.0f}원")

        investable_krw = min(self.max_investment, krw_balance)
        print(f"투자 설정액: {self.max_investment:,.0f}원, 실제 투자 가능액: {investable_krw:,.0f}원")

        if investable_krw < self.min_order_amount:
            print(f"⚠️ 투자 가능액({investable_krw:,.0f}원)이 최소 주문액({self.min_order_amount:,.0f}원)보다 작습니다.")
            return 0

        if price_per_btc <= 0:
            print("⚠️ BTC 가격이 0 이하이므로 수량 계산 불가.")
            return 0

        # BTC 구매 수량 계산 (수수료 미고려)
        btc_amount = investable_krw / price_per_btc
        return round(btc_amount, 8)

    def run_trading_loop(self, interval=60, max_iterations=None): # 수정됨
        """거래 루프 실행"""
        print("🚀 자동거래 시작!")
        print(f"거래 페어: {self.currency_pair.upper()}")
        print(f"최대 투자 설정액 (1회): {self.max_investment:,.0f}원")
        print(f"최소 주문 설정액 (1회): {self.min_order_amount:,.0f}원")
        print(f"수익 실현 목표: {self.profit_target*100:.2f}%")
        print(f"손절매 기준: {self.stop_loss*100:.2f}%")
        print(f"체크 간격: {interval}초")
        if max_iterations:
            print(f"최대 실행 횟수: {max_iterations}회")
        print("-" * 50)

        iteration = 0
        while True:
            iteration += 1
            now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            print(f"\n⏰ {now_str} (실행 {iteration}회차)")

            if max_iterations and iteration > max_iterations:
                print(f"\n⏹️ 최대 실행 횟수({max_iterations}회)에 도달하여 거래를 종료합니다.")
                break

            try:
                ticker_data = self.get_ticker()
                if not ticker_data:
                    print("⚠️ Ticker 정보 조회 실패. 다음 주기로 넘어갑니다.")
                    time.sleep(interval)
                    continue

                current_price_str = ticker_data.get('close')
                if current_price_str is None:
                    print(f"❌ Ticker 데이터에서 'close' 키를 찾을 수 없습니다. 다음 주기로 넘어갑니다. 데이터: {ticker_data}")
                    time.sleep(interval)
                    continue
                try:
                    current_price = float(current_price_str)
                except ValueError:
                    print(f"❌ Ticker 'close' 값 ({current_price_str})을 숫자로 변환 불가. 다음 주기로 넘어갑니다.")
                    time.sleep(interval)
                    continue

                # 거래 전략 실행
                signal = self.simple_strategy() # simple_strategy 내부에서도 get_ticker 호출. 비효율적일 수 있음.

                if signal:
                    print(f"📊 거래 신호: {signal} (현재가: {current_price:,.0f}원)")

                    if signal == "BUY":
                        has_btc_position = False
                        balance = self.get_balance()
                        if balance:
                            for acc in balance:
                                if acc.get('currency') == 'btc' and float(acc.get('available', 0)) > 0.00000001: # 극소량 이상 보유시
                                    has_btc_position = True
                                    print(f"ℹ️ 이미 BTC 보유 중 ({float(acc.get('available', 0))} BTC). 추가 매수하지 않음.")
                                    break

                        if not has_btc_position:
                            btc_to_buy = self.calculate_position_size(current_price)
                            if btc_to_buy > 0:
                                print(f"💰 매수 주문 시도: {btc_to_buy:.8f} BTC at market price (약 {current_price:,.0f}원)")
                                order_result = self.place_order(side='buy', qty=btc_to_buy) # 시장가 매수
                                if order_result:
                                    self.positions.append({
                                        'type': 'BUY', 'qty': btc_to_buy, 'price': current_price,
                                        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                                        'order_id': order_result.get('orderId') # API 응답에서 주문 ID 가져오기
                                    })
                                    print(f"✅ 매수 주문 결과: {order_result}")
                            else:
                                print("ℹ️ 계산된 BTC 매수 수량이 0 이하입니다. 매수하지 않음.")

                    elif signal == "SELL":
                        balance = self.get_balance()
                        btc_available = 0
                        if balance:
                            for account in balance:
                                if account.get('currency') == 'btc':
                                    try:
                                        btc_available = float(account.get('available', 0))
                                    except ValueError:
                                        btc_available = 0
                                    break

                        if btc_available > 0.00000001: # 극소량 이상 있을 경우 매도
                            print(f"💰 매도 주문 시도: {btc_available:.8f} BTC at market price (약 {current_price:,.0f}원)")
                            order_result = self.place_order(side='sell', qty=btc_available) # 시장가 매도
                            if order_result:
                                # 포지션 정리 로직 (간단화)
                                self.positions = [p for p in self.positions if p['type'] != 'BUY'] # 모든 매수 포지션 제거
                                self.positions.append({
                                    'type': 'SELL', 'qty': btc_available, 'price': current_price,
                                    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                                    'order_id': order_result.get('orderId')
                                })
                                print(f"✅ 매도 주문 결과: {order_result}")
                        else:
                            print("ℹ️ 매도할 BTC 잔고가 없습니다.")
                else:
                    print("📊 거래 신호: HOLD 또는 데이터 부족")

                # 현재 잔고 표시 (매 루프마다 표시)
                current_balances = self.get_balance()
                if current_balances:
                    print("💰 현재 잔고:")
                    for acc_bal in current_balances:
                        try:
                            avail_bal = float(acc_bal.get('available',0))
                            if avail_bal > 0 : # 0보다 큰 잔고만 표시 (문자열 0.0도 포함)
                                print(f"  {acc_bal.get('currency','N/A').upper()}: {avail_bal:,.8f}")
                        except ValueError:
                             print(f"  {acc_bal.get('currency','N/A').upper()}: (값 변환 불가 - {acc_bal.get('available')})")


            except KeyboardInterrupt:
                print("\n🛑 사용자에 의해 거래가 중단되었습니다.")
                break
            except Exception as e:
                print(f"❌ 거래 루프 중 예기치 않은 오류 발생: {e}")
                import traceback
                traceback.print_exc() # 상세 오류 스택 출력
                print("오류 발생으로 인해 30초 후 재시도합니다...")
                time.sleep(30) # 오류 발생 시 잠시 대기 후 재시도

            print(f"💤 {interval}초 후 다음 사이클 시작...")
            time.sleep(interval)

# 코랩용 사용 예시
def main(): # 수정됨
    from getpass import getpass

    print("🔐 Korbit API 정보를 입력해주세요:")
    API_KEY = getpass("API Key: ")
    API_SECRET = getpass("API Secret: ")

    trader = KorbitTrader(API_KEY, API_SECRET)

    if trader.test_connection():
        print("\n📊 현재 시장 상황:")
        ticker = trader.get_ticker()
        if ticker:
            try:
                current_price_str = ticker.get('close')
                change_pct_str = ticker.get('priceChangePercent')

                if current_price_str is not None and change_pct_str is not None:
                    print(f"BTC/KRW: {float(current_price_str):,.0f}원")
                    print(f"24시간 변화: {float(change_pct_str):+.2f}%")
                else:
                    print("Ticker 정보에서 'close' 또는 'priceChangePercent' 키를 찾을 수 없습니다.")
                print(f"전체 Ticker 정보: {ticker}")

            except KeyError as e:
                print(f"❌ Ticker 데이터에서 필요한 키를 찾을 수 없습니다: {e}")
                print(f"받은 Ticker 데이터: {ticker}")
            except ValueError as e:
                print(f"❌ Ticker 데이터의 숫자 변환 중 오류 발생: {e}")
                print(f"받은 Ticker 데이터: {ticker}")
            except Exception as e:
                print(f"❌ Ticker 정보 처리 중 알 수 없는 오류: {e}")


        balance_data = trader.get_balance()
        if balance_data:
            print("\n💰 현재 잔고:")
            for account in balance_data:
                try:
                    available_balance = float(account.get('available', 0))
                    if available_balance > 0: # 0보다 큰 잔고만 표시
                         print(f"  {account.get('currency','N/A').upper()}: {available_balance:,.8f}")
                except ValueError:
                    print(f"  {account.get('currency','N/A').upper()}: 잔액을 숫자로 변환할 수 없습니다 - {account.get('available')}")
                except KeyError: # .get() 사용으로 KeyError는 잘 발생 안 함
                    print(f"  {account.get('currency','N/A').upper()}: 필수 키를 찾을 수 없습니다.")
        else:
            print("잔고 정보를 가져올 수 없습니다.")


        print("\n📋 실행 옵션:")
        print("1. 무한 실행 (Ctrl+C로 중단)")
        print("2. 제한된 횟수 실행 (코랩 세션 관리용)")

        choice = input("선택 (1 또는 2): ")

        if choice == "2":
            try:
                max_iter_input = input("최대 실행 횟수: ")
                max_iter = int(max_iter_input)
                if max_iter <=0:
                    print("최대 실행 횟수는 0보다 커야 합니다. 기본값(10회)으로 실행합니다.")
                    max_iter = 10
                trader.run_trading_loop(interval=60, max_iterations=max_iter)
            except ValueError:
                print("❌ 잘못된 입력입니다. 숫자를 입력해주세요. 기본값(10회)으로 실행합니다.")
                trader.run_trading_loop(interval=60, max_iterations=10) # 예외 발생시 기본값으로 실행
        elif choice == "1":
            trader.run_trading_loop(interval=60)
        else:
            print("잘못된 선택입니다. 프로그램을 종료합니다.")

    else:
        print("❌ API 연결에 실패했습니다. API 키와 시크릿, 네트워크 상태를 확인해주세요.")
        print("\n💡 확인사항:")
        print("1. API 키/시크릿이 올바른지 확인")
        print("2. API 키에 거래 권한이 있는지 확인 (Korbit 웹사이트)")
        print("3. Korbit API 서버 상태 확인 (공지사항 등)")
        print("4. 사용자의 네트워크 연결 상태 확인")


if __name__ == "__main__":
    main()